In [18]:
import numpy as np
from numpy.random import rand, randint
import gurobipy as gp
from gurobipy import GRB
from sympy.utilities.iterables import multiset_permutations
from TakeTwoSetup import *

# --- Finding D and H --- #
# D: max number of days
# H: max number of scenes in a day
def find_D(n, S, durations, W):
  D = (2*n*(max(durations)+np.matrix.max(S)))/(W+np.matrix.max(S))
  if np.ceil(D)%2 == 0:
    D = int(np.floor(D))
  else:
    D=int(np.ceil(D))
  return D

D = find_D(n, S, durations, W)

def find_H(n, S, durations, W):
  increasingDurations = np.sort(durations)
  increasingChangeover = np.array(np.sort(np.matrix.flatten(S)))[0]

  vals = []
  hs = []
  for h in range(2, n+1):
    total = np.sum(increasingChangeover[:h]) + np.sum(increasingDurations[:h])
    if total <= W:
      # to make sure combination is within admissible set
      vals.append(total)
      hs.append(h)
  H = hs[np.argmax(vals)]
  return H

H = find_H(n, S, durations, W)

D, H

(9, 4)

In [19]:
setOfIdx = set([])
def finding_P(n, S, durations, W, H):
    perm = []
    for i in range(H+1):
        perm.extend(multiset_permutations(range(n), i))
    
    for p in perm:
        total = 0
        for i in range(len(p)-1):
            total += S[p[i], p[i+1]]
        for i in p:
            total += durations[i]
        if total > W:
            perm.remove(p)
    return perm

P = finding_P(n, S, durations, W, H)[1:]
k = len(P)
k, P

(1889,
 [[0],
  [1],
  [2],
  [3],
  [4],
  [5],
  [6],
  [7],
  [8],
  [0, 1],
  [0, 2],
  [0, 3],
  [0, 4],
  [0, 5],
  [0, 6],
  [0, 7],
  [0, 8],
  [1, 0],
  [1, 2],
  [1, 3],
  [1, 4],
  [1, 5],
  [1, 6],
  [1, 7],
  [1, 8],
  [2, 0],
  [2, 1],
  [2, 3],
  [2, 4],
  [2, 5],
  [2, 6],
  [2, 7],
  [2, 8],
  [3, 0],
  [3, 1],
  [3, 2],
  [3, 4],
  [3, 5],
  [3, 6],
  [3, 7],
  [3, 8],
  [4, 0],
  [4, 1],
  [4, 2],
  [4, 3],
  [4, 5],
  [4, 6],
  [4, 7],
  [4, 8],
  [5, 0],
  [5, 1],
  [5, 2],
  [5, 3],
  [5, 4],
  [5, 6],
  [5, 7],
  [5, 8],
  [6, 0],
  [6, 1],
  [6, 2],
  [6, 3],
  [6, 4],
  [6, 5],
  [6, 7],
  [6, 8],
  [7, 0],
  [7, 1],
  [7, 2],
  [7, 3],
  [7, 4],
  [7, 5],
  [7, 6],
  [7, 8],
  [8, 0],
  [8, 1],
  [8, 2],
  [8, 3],
  [8, 4],
  [8, 5],
  [8, 6],
  [8, 7],
  [0, 1, 3],
  [0, 1, 5],
  [0, 1, 7],
  [0, 2, 1],
  [0, 2, 4],
  [0, 2, 6],
  [0, 2, 8],
  [0, 3, 2],
  [0, 3, 5],
  [0, 3, 6],
  [0, 3, 8],
  [0, 4, 2],
  [0, 4, 5],
  [0, 4, 6],
  [0, 4, 8],
  [0, 5, 2],
  

In [61]:
# --- ILP_pat Model Setup --- #
ILP_pat = gp.Model("ILP_pat", env=env)

# decision variables
a = ILP_pat.addVars(k, n,vtype=GRB.BINARY, name="a")
x_pat = ILP_pat.addVars(k, D, vtype=GRB.BINARY, name="x_pat")
y_pat = ILP_pat.addVars(m, D, D, vtype=GRB.BINARY, name="y_pat")

# objective function
objFunc_pat = ILP_pat.setObjective(
    gp.quicksum(
        holdingCost[i] *
        gp.quicksum(
              (d2-d1+1)*y_pat[i, d1, d2]
              for d1 in range(D)
              for d2 in range(d1, D)
        )
        for i in range(len(actors))
    ), GRB.MINIMIZE
)

# --- constraints (lord help us) --- #
c21 = ILP_pat.addConstr(
    gp.quicksum(
        x_pat[k, 1] for k in range(len(P))
    ) == 1, name="c21"
) # Constraint (21)

c22 = ILP_pat.addConstrs((
    gp.quicksum(
        x_pat[k, d-1]
        for k in range(len(P))
        ) <=
    gp.quicksum(
        x_pat[k, d]
        for k in range(len(P))
        )
    for d in range(2, D)),
    name="c22"
) # Constraint (22)

c23 = ILP_pat.addConstrs((
    gp.quicksum(
        a[k, j] * x_pat[k, d]
        for k in range(len(P))
        for d in range(D)
    ) == 1
    for j in range(n)),
    name="c23"
) # Constraint (23)

c24 = ILP_pat.addConstrs((
    gp.quicksum(
        y_pat[i, d1, d2]
        for d1 in range(D)
        for d2 in range(d1, D)
    ) == 1
    for i in range(m)),
    name="c24"
) # Constraint (24)

c25 = ILP_pat.addConstrs((
    gp.quicksum(
        T[i, j] * a[k, j] * x_pat[k, d]
        for k in range(len(P))
        for j in range(n)
    ) <=
    gp.quicksum(
        y_pat[i, d1, d2]
        for d2 in range(d, D)
        for d1 in range(d)
    )
    for i in range(m) for d in range(D)),
    name="c25"
) # Constraint (25)

# --- Constraints (26-27) --- #
# These constraints are embedded in the init of the variables

In [62]:
# --- optimize that shit --- #
ILP_pat.setAttr()
ILP_pat.optimize()
sol = ILP_pat.getJSONSolution()

Set parameter TimeLimit to value 300


Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i5-9600KF CPU @ 3.70GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 6 logical processors, using up to 6 threads

Non-default parameters:
TimeLimit  300

Academic license 2773076 - for non-commercial use only - registered to km___@uga.edu
Optimize a model with 17 rows, 34749 columns and 28755 nonzeros (Min)
Model fingerprint: 0x10e6d6ea
Model has 405 linear objective coefficients
Model has 90 quadratic constraints
Variable types: 0 continuous, 34749 integer (34749 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [6e+02, 8e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
  QRHS range       [1e+00, 1e+00]

Presolve added 21 rows and 0 columns
Presolve removed 0 rows and 382 columns
Presolve time: 1.05s
Presolved: 459389 rows, 187457

In [63]:
all_vars = ILP_pat.getVars()
values = ILP_pat.getAttr("X", all_vars)
names = ILP_pat.getAttr("VarName", all_vars)

for name, val in zip(names, values):
    if val == 1:
        print(f"{name} = {val}")

a[4,1] = 1.0
a[13,1] = 1.0
a[26,5] = 1.0
a[34,7] = 1.0
a[36,2] = 1.0
a[65,8] = 1.0
a[69,5] = 1.0
a[73,7] = 1.0
a[75,5] = 1.0
a[75,7] = 1.0
a[81,3] = 1.0
a[91,3] = 1.0
a[105,7] = 1.0
a[109,5] = 1.0
a[121,1] = 1.0
a[127,6] = 1.0
a[131,3] = 1.0
a[142,3] = 1.0
a[143,1] = 1.0
a[168,8] = 1.0
a[170,3] = 1.0
a[171,0] = 1.0
a[171,8] = 1.0
a[176,7] = 1.0
a[188,2] = 1.0
a[193,4] = 1.0
a[193,8] = 1.0
a[195,4] = 1.0
a[203,0] = 1.0
a[206,7] = 1.0
a[215,5] = 1.0
a[243,1] = 1.0
a[251,3] = 1.0
a[255,7] = 1.0
a[256,4] = 1.0
a[258,7] = 1.0
a[290,0] = 1.0
a[292,0] = 1.0
a[319,8] = 1.0
a[322,0] = 1.0
a[329,8] = 1.0
a[331,1] = 1.0
a[338,3] = 1.0
a[341,0] = 1.0
a[350,2] = 1.0
a[354,8] = 1.0
a[356,1] = 1.0
a[359,5] = 1.0
a[359,8] = 1.0
a[362,2] = 1.0
a[363,6] = 1.0
a[391,8] = 1.0
a[397,2] = 1.0
a[405,8] = 1.0
a[421,8] = 1.0
a[425,4] = 1.0
a[432,7] = 1.0
a[437,8] = 1.0
a[453,1] = 1.0
a[492,4] = 1.0
a[518,4] = 1.0
a[535,6] = 1.0
a[536,2] = 1.0
a[545,3] = 1.0
a[550,8] = 1.0
a[554,1] = 1.0
a[556,0] = 1.0
a[574,2]

In [64]:
sol

'{ "SolutionInfo": { "Status": 9, "Runtime": 3.0009699988365173e+02, "Work": 2.8189305916011438e+02, "ObjVal": 42100, "ObjBound": 39390, "ObjBoundC": 3.9389999999999993e+04, "MIPGap": 6.4370546318289784e-02, "IntVio": 0, "BoundVio": 0, "ConstrVio": 0, "IterCount": 204771, "BarIterCount": 0, "NLBarIterCount": 0, "PDHGIterCount": 0, "NodeCount": 1, "SolCount": 4, "PoolObjBound": 39390, "PoolNObjVal": [ 42100, 45510, 45670, 51340]}, "Vars": [ { "VarName": "a[4,1]", "X": 1}, { "VarName": "a[13,1]", "X": 1}, { "VarName": "a[26,5]", "X": 1}, { "VarName": "a[34,7]", "X": 1}, { "VarName": "a[36,2]", "X": 1}, { "VarName": "a[65,8]", "X": 1}, { "VarName": "a[69,5]", "X": 1}, { "VarName": "a[73,7]", "X": 1}, { "VarName": "a[75,5]", "X": 1}, { "VarName": "a[75,7]", "X": 1}, { "VarName": "a[81,3]", "X": 1}, { "VarName": "a[91,3]", "X": 1}, { "VarName": "a[105,7]", "X": 1}, { "VarName": "a[109,5]", "X": 1}, { "VarName": "a[121,1]", "X": 1}, { "VarName": "a[127,6]", "X": 1}, { "VarName": "a[131,3]", 